# 08 Tree-based models

Random forest and gradient boosted regression trees (LightGBM) on the parsimonious predictor set of the `_mv25` samples, with the rolling windows and the hyperparameter grids of Drobetz and Otto (2021, Table 9); the forest uses 300 trees grown on 63.2% subsamples drawn without replacement. Forecasts are joined with those of the linear models (notebook 07). Each test year is saved as a checkpoint, so an interrupted run resumes from the first missing year. Sections 3.6.2, 3.6.3 and 3.7, Chapter 4.

## Configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, time, gc, json, glob
import numpy as np
import pandas as pd

EST_DIR   = "/content/drive/MyDrive/Thesis/data"   # estimation samples (notebook 6)
PANEL_DIR = "/content/drive/MyDrive/Thesis/data"   # panel_characteristics.parquet (notebook 5)
LIN_DIR   = "/content/drive/MyDrive/Thesis/data"   # predictions_linear_*.parquet (notebook 7)
OUT_DIR   = "/content/drive/MyDrive/Thesis/data"   # results of this notebook
CKPT_DIR  = os.path.join(OUT_DIR, "trees_checkpoints")

SAMPLES = ["complete_case_mv25", "imputed_mv25"]   # complete-case first, imputed second
FIRST_TEST, LAST_TEST = 2010, 2025
N_TRAIN, N_VAL = 8, 2
SEED = 0

RF_DEPTHS, RF_MTRY, RF_TREES = range(1, 7), (22, 33, 44), 300
GB_DEPTHS, GB_LR = (1, 2, 3), (0.01, 0.05, 0.1)
GB_MAX_TREES, GB_MIN_TREES, GB_PATIENCE = 1000, 10, 200

## Functions

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
import lightgbm as lgb

warnings.filterwarnings("ignore", category=UserWarning)

CHARS = ["size", "bm", "operatingprofitability", "totalassetgrowth", "ret_2_12",
         "issues_1_36", "accrualschange", "roa", "investment", "ret_1",
         "ret_12_36", "dy", "beta", "vola", "turnover", "debttoprice",
         "salestoprice", "cftoprice", "earningstoprice", "issues_1_12",
         "fcdispersion", "grossprofitability"]

MODELS = ["ols3", "ols_pars", "ols_full", "elanet", "rf", "gbrt"]
LINEAR = ["ols3", "ols_pars", "ols_full", "elanet"]

# Drobetz and Otto (2021), Table 2, test period 2000-2020
DO_R2 = {"ols_full": -0.52, "ols_pars": 0.57, "elanet": 1.21, "rf": 1.18, "gbrt": 1.14}
DO_R2_LARGE = {"ols_full": -1.16, "ols_pars": 0.11, "elanet": 0.75, "rf": 0.92, "gbrt": 0.74}
DO_R2_SMALL = {"ols_full": -0.10, "ols_pars": 0.90, "elanet": 1.55, "rf": 1.40, "gbrt": 1.46}
DO_PS = {"ols_full": 0.45, "ols_pars": 0.65, "elanet": 0.92, "rf": 1.03, "gbrt": 1.07}


# Data

def load_sample(name, est_dir, panel_dir):
    est = pd.read_parquet(os.path.join(est_dir, f"estimation_{name}.parquet"))
    est["date"] = pd.to_datetime(est["date"])
    raw = pd.read_parquet(os.path.join(panel_dir, "panel_characteristics.parquet"),
                          columns=["symbol", "date", "target"])
    raw["date"] = pd.to_datetime(raw["date"])
    return (est.merge(raw.rename(columns={"target": "raw_target"}), on=["symbol", "date"],
                      how="left").sort_values(["date", "symbol"]).reset_index(drop=True))


def build_parsimonious(df, chars=CHARS):
    """22 rank-transformed characteristics plus country and sector indicators
    (one category of each dropped), as for the linear `ols_pars` model."""
    dummies = pd.concat([
        pd.get_dummies(df["country"], prefix="cty", drop_first=True, dtype=np.float32),
        pd.get_dummies(df["sector"], prefix="sec", drop_first=True, dtype=np.float32)], axis=1)
    X = np.hstack([df[chars].to_numpy(dtype=np.float32), dummies.to_numpy()])
    return X, list(chars) + list(dummies.columns)


def rolling_splits(dates, first_test_year, last_test_year, n_train=8, n_val=2):
    years = pd.DatetimeIndex(dates).year
    for y in range(first_test_year, last_test_year + 1):
        tr = (years >= y - n_val - n_train) & (years <= y - n_val - 1)
        va = (years >= y - n_val) & (years <= y - 1)
        te = years == y
        if te.sum() and tr.sum() and va.sum():
            yield y, np.asarray(tr), np.asarray(va), np.asarray(te)


# Models

def base_params(seed, threads):
    return {"objective": "regression", "verbose": -1, "seed": seed, "deterministic": True,
            "force_row_wise": True, "num_threads": threads, "min_data_in_leaf": 20}


def fit_rf(dtr, Xva, yva, n_features, depths, mtries, n_trees, seed=0, threads=0):
    """
    Random forest tuned on the validation sample over tree depth and the number of
    predictors drawn at each split (Drobetz and Otto, Table 9). Implemented in
    LightGBM: each of the n_trees trees is grown on a random 63.2% subsample drawn
    without replacement, the expected share of distinct observations in a bootstrap
    sample, and considers a random subset of `mtry` predictors at each split.
    """
    best = None
    for d in depths:
        for m in mtries:
            p = {**base_params(seed, threads), "boosting": "rf", "num_leaves": 2 ** d,
                 "max_depth": d, "bagging_freq": 1, "bagging_fraction": 0.632,
                 "feature_fraction_bynode": min(1.0, m / n_features)}
            mdl = lgb.train(p, dtr, num_boost_round=n_trees)
            mse = float(np.mean((yva - mdl.predict(Xva)) ** 2))
            if best is None or mse < best["val_mse"]:
                best = {"val_mse": mse, "depth": d, "mtry": m, "model": mdl}
    return best


def fit_gbrt(dtr, dva, depths, lrs, max_trees, min_trees=10, patience=200, seed=0, threads=0):
    """
    Gradient boosted regression trees tuned on the validation sample over tree depth,
    learning rate and number of trees (Drobetz and Otto, Table 9). The validation
    error is recorded after every tree; the number of trees is the one that minimises
    it, between `min_trees` and `max_trees`, with training stopped once it has not
    improved for `patience` trees.
    """
    best = None
    for d in depths:
        for lr in lrs:
            ev = {}
            p = {**base_params(seed, threads), "boosting": "gbdt", "learning_rate": lr,
                 "num_leaves": 2 ** d, "max_depth": d}
            mdl = lgb.train(p, dtr, num_boost_round=max_trees, valid_sets=[dva], valid_names=["val"],
                            callbacks=[lgb.record_evaluation(ev),
                                       lgb.early_stopping(patience, first_metric_only=True, verbose=False)])
            l2 = np.asarray(ev["val"]["l2"])
            lo = min(min_trees, len(l2)) - 1
            k = lo + int(np.argmin(l2[lo:]))
            if best is None or l2[k] < best["val_mse"]:
                best = {"val_mse": float(l2[k]), "depth": d, "lr": lr, "n_trees": k + 1, "model": mdl}
    return best


# Evaluation

def r2_oos(y, yhat):
    y, yhat = np.asarray(y), np.asarray(yhat)
    return 1.0 - np.sum((y - yhat) ** 2) / np.sum(y ** 2)


def r2_cross_section(df, m, y="target"):
    yy = df[y] - df.groupby("date")[y].transform("mean")
    ff = df[m] - df.groupby("date")[m].transform("mean")
    return 1.0 - ((yy - ff) ** 2).sum() / (yy ** 2).sum()


def predictive_slope(y, yhat):
    yhat = np.asarray(yhat)
    v = np.var(yhat)
    return np.cov(np.asarray(y), yhat, bias=True)[0, 1] / v if v > 0 else np.nan


def newey_west_se(x, lags=None):
    x = np.asarray(x, dtype=float)
    T = len(x)
    if lags is None:
        lags = int(np.floor(4 * (T / 100) ** (2 / 9)))
    u = x - x.mean()
    s = np.mean(u ** 2)
    for l in range(1, lags + 1):
        s += 2 * (1 - l / (lags + 1)) * np.mean(u[l:] * u[:-l])
    return np.sqrt(s / T)


def diebold_mariano(df, model_a, model_b, y="target"):
    """Positive: model_b more accurate than model_a."""
    d = ((df[y] - df[model_a]) ** 2 - (df[y] - df[model_b]) ** 2).groupby(df["date"]).mean()
    return d.mean() / newey_west_se(d.values)


def decile_portfolios(df, pred, ret="raw_target", weight="MV"):
    d = df.dropna(subset=[pred, ret]).copy()
    d["dec"] = d.groupby("date")[pred].transform(
        lambda s: pd.qcut(s.rank(method="first"), 10, labels=False))
    ew = d.groupby(["date", "dec"])[ret].mean().unstack()
    d["_w"] = d[weight].clip(lower=0)
    d["_wr"] = d["_w"] * d[ret]
    g = d.groupby(["date", "dec"])
    vw = (g["_wr"].sum(min_count=1) / g["_w"].sum(min_count=1)).unstack()
    return ew, vw


def portfolio_stats(port):
    hl = (port[9] - port[0]).dropna()
    return {"mean_%": 100 * hl.mean(), "t_stat": hl.mean() / newey_west_se(hl.values),
            "sharpe_ann": hl.mean() / hl.std() * np.sqrt(12), "months": len(hl)}


def annual_r2(P, models, y="target"):
    return pd.DataFrame({m: P.groupby(P["date"].dt.year).apply(
        lambda g: 100 * r2_oos(g[y], g[m]), include_groups=False) for m in models})


def periods_of(P):
    yr = P["date"].dt.year
    return {"2010-2025": yr >= 2010, "2010-2020": yr <= 2020, "2021-2025": yr >= 2021}


def avg_annual_table(P, models, y="target"):
    a = annual_r2(P, models, y)
    return pd.DataFrame({"2010-2025": a.mean(), "2010-2020": a.loc[:2020].mean(),
                         "2021-2025": a.loc[2021:].mean()})


def side_by_side(tables, ref=None, ref_label="DO"):
    out = pd.concat(tables, axis=1)
    if ref is not None:
        out[(ref_label, "2000-2020")] = pd.Series(ref)
    return out


def group_importance(imp, names):
    """Aggregate gain importance: each characteristic, plus all country and all sector indicators."""
    s = pd.Series(imp, index=names)
    out = s[[n for n in names if not n.startswith(("cty_", "sec_"))]].copy()
    out["country (all)"] = s[[n for n in names if n.startswith("cty_")]].sum()
    out["sector (all)"] = s[[n for n in names if n.startswith("sec_")]].sum()
    return out / out.sum()


## Estimation

In [ ]:
t0 = time.time()
for s in SAMPLES:
    ck = os.path.join(CKPT_DIR, s)
    os.makedirs(ck, exist_ok=True)
    est = load_sample(s, EST_DIR, PANEL_DIR)
    X, names = build_parsimonious(est)
    y = est["target"].to_numpy()
    with open(os.path.join(ck, "names.json"), "w") as fh:
        json.dump(names, fh)
    print(f"\n{s}: {len(est):,} security-months, {X.shape[1]} predictors")

    for yr, tr, va, te in rolling_splits(est["date"], FIRST_TEST, LAST_TEST, N_TRAIN, N_VAL):
        f_pred = os.path.join(ck, f"{yr}_pred.parquet")
        if os.path.exists(f_pred):
            print(f"   {yr}: already estimated, skipped")
            continue
        t1 = time.time()
        dtr = lgb.Dataset(X[tr], y[tr], params={"verbose": -1}, free_raw_data=False).construct()
        dva = lgb.Dataset(X[va], y[va], reference=dtr).construct()

        rf = fit_rf(dtr, X[va], y[va], X.shape[1], RF_DEPTHS, RF_MTRY, RF_TREES, SEED)
        gb = fit_gbrt(dtr, dva, GB_DEPTHS, GB_LR, GB_MAX_TREES, GB_MIN_TREES, GB_PATIENCE, SEED)

        out = est.loc[te, ["symbol", "date", "target", "raw_target", "MV", "country"]].copy()
        out["rf"] = rf["model"].predict(X[te])
        out["gbrt"] = gb["model"].predict(X[te], num_iteration=gb["n_trees"])

        pd.DataFrame({"name": names,
                      "rf": rf["model"].feature_importance("gain"),
                      "gbrt": gb["model"].feature_importance("gain", iteration=gb["n_trees"])}
                     ).to_parquet(os.path.join(ck, f"{yr}_imp.parquet"), index=False)
        with open(os.path.join(ck, f"{yr}_tuning.json"), "w") as fh:
            json.dump({"year": yr, "rf_depth": int(rf["depth"]), "rf_mtry": int(rf["mtry"]),
                       "gbrt_depth": int(gb["depth"]), "gbrt_lr": float(gb["lr"]),
                       "gbrt_trees": int(gb["n_trees"])}, fh)
        out.to_parquet(f_pred, index=False)          # written last: marks the year complete

        print(f"   {yr}: rf depth {rf['depth']}, mtry {rf['mtry']} | gbrt depth {gb['depth']}, "
              f"lr {gb['lr']}, {gb['n_trees']} trees | {time.time()-t1:.0f}s")
    del est, X, y; gc.collect()
print(f"\nDone in {(time.time()-t0)/60:.1f} min.")

## Assembling the forecasts

In [ ]:
R = {}
for s in SAMPLES:
    ck = os.path.join(CKPT_DIR, s)
    Pt = pd.concat([pd.read_parquet(f) for f in sorted(glob.glob(os.path.join(ck, "*_pred.parquet")))],
                   ignore_index=True)
    lin = pd.read_parquet(os.path.join(LIN_DIR, f"predictions_linear_{s}.parquet"))
    keep = ["symbol", "date"] + LINEAR + (["n_imputed"] if "n_imputed" in lin.columns else [])
    lin["date"] = pd.to_datetime(lin["date"])
    P = Pt.merge(lin[keep], on=["symbol", "date"], how="left")
    T = pd.DataFrame([json.load(open(f)) for f in sorted(glob.glob(os.path.join(ck, "*_tuning.json")))])
    names = json.load(open(os.path.join(ck, "names.json")))
    imp = {m: pd.DataFrame({int(os.path.basename(f)[:4]): group_importance(pd.read_parquet(f)[m].values, names)
                            for f in sorted(glob.glob(os.path.join(ck, "*_imp.parquet")))})
           for m in ["rf", "gbrt"]}
    R[s] = {"P": P, "T": T, "imp": imp}
    miss = int(P[LINEAR].isna().any(axis=1).sum())
    print(f"{s:20s} {len(P):>9,} forecasts, {P['date'].dt.year.nunique()} test years, "
          f"linear forecasts missing for {miss} rows")

## Each sample on its own observations

In [ ]:
avg = side_by_side({s: avg_annual_table(R[s]["P"], MODELS) for s in SAMPLES}, DO_R2)
print("Average of annual R^2 (%):")
print(avg.round(3).to_string())

pooled = pd.concat({s: pd.DataFrame({p: {m: 100 * r2_oos(R[s]["P"].loc[mk, "target"], R[s]["P"].loc[mk, m])
                                         for m in MODELS} for p, mk in periods_of(R[s]["P"]).items()})
                    for s in SAMPLES}, axis=1)
print("\nPooled R^2 (%):")
print(pooled.round(3).to_string())

In [ ]:
for s in SAMPLES:
    print(f"\nR^2 (%) by test year -- {s}:")
    print(annual_r2(R[s]["P"], MODELS).round(2).to_string())

### Ranking stocks versus forecasting the market

In [ ]:
cs = pd.concat({s: pd.DataFrame({p: {m: 100 * r2_cross_section(R[s]["P"][mk], m) for m in MODELS}
                                  for p, mk in periods_of(R[s]["P"]).items()}) for s in SAMPLES}, axis=1)
print("Cross-sectional R^2 (%):")
print(cs.round(3).to_string())

corr = {}
for s in SAMPLES:
    P = R[s]["P"]
    mkt = P.groupby("date")["target"].mean().groupby(lambda d: d.year).mean()
    corr[s] = annual_r2(P, MODELS).corrwith(mkt)
print("\nCorrelation of annual R^2 with the year's average excess return:")
print(pd.DataFrame(corr).round(2).to_string())

### Large and small firms

In [ ]:
size_tabs = {}
for s in SAMPLES:
    P = R[s]["P"]
    large = P["MV"] >= P.groupby("date")["MV"].transform("median")
    size_tabs[s] = pd.DataFrame({"large": annual_r2(P[large], MODELS).mean(),
                                 "small": annual_r2(P[~large], MODELS).mean()})
size_r2 = pd.concat(size_tabs, axis=1)
size_r2[("DO", "large")] = pd.Series(DO_R2_LARGE)
size_r2[("DO", "small")] = pd.Series(DO_R2_SMALL)
print("Average of annual R^2 (%) by size, 2010-2025:")
print(size_r2.round(3).to_string())

### Predictive slopes

Drobetz and Otto report 1.03 for the random forest and 1.07 for gradient boosting.

In [ ]:
ps = {}
for s in SAMPLES:
    P = R[s]["P"]
    a = pd.DataFrame({m: P.groupby(P["date"].dt.year).apply(
        lambda g: predictive_slope(g["target"], g[m]), include_groups=False) for m in MODELS})
    ps[s] = pd.DataFrame({"2010-2025": a.mean(), "2010-2020": a.loc[:2020].mean(),
                          "2021-2025": a.loc[2021:].mean()})
    unstable = a.stack()[a.stack().abs() > 5]
    if len(unstable):
        print(f"{s}: unstable slopes (|PS| > 5): "
              + ", ".join(f"{m} {y}: {v:.1f}" for (y, m), v in unstable.items()))
ps_tab = side_by_side(ps, DO_PS)
print(ps_tab.round(3).to_string())

### Diebold-Mariano tests

Positive: the column model is more accurate than the row model; |DM| > 1.96 is
significant at 5%. The rows `rf` and `gbrt` against the linear columns, and vice
versa, show whether the trees improve on the linear models.

In [ ]:
dm = {}
for s in SAMPLES:
    d = pd.DataFrame(index=MODELS, columns=MODELS, dtype=float)
    for a in MODELS:
        for b in MODELS:
            if a != b:
                d.loc[a, b] = diebold_mariano(R[s]["P"], a, b)
    dm[s] = d
    print(f"\n{s}:")
    print(d.round(2).to_string())

## Comparing the samples

### On the same observations

In [ ]:
CMP = ["ols3", "elanet", "rf", "gbrt"]
Pc, Pi = R[SAMPLES[0]]["P"], R[SAMPLES[1]]["P"]
common = Pc[["symbol", "date", "target"] + CMP].merge(
    Pi[["symbol", "date"] + CMP].rename(columns={m: f"{m}_imp" for m in CMP}),
    on=["symbol", "date"], how="inner")
print(f"Common observations: {len(common):,} of {len(Pc):,} complete-case forecasts\n")

rows = []
yr = common["date"].dt.year
for m in CMP:
    rows.append({"model": m,
                 "R2 trained CC": annual_r2(common, [m]).iloc[:, 0].mean(),
                 "R2 trained imputed": annual_r2(common.rename(columns={f"{m}_imp": "x"}), ["x"]).iloc[:, 0].mean(),
                 "CS R2 trained CC": 100 * r2_cross_section(common, m),
                 "CS R2 trained imputed": 100 * r2_cross_section(common, f"{m}_imp"),
                 "DM 2010-2025": diebold_mariano(common, m, f"{m}_imp"),
                 "DM 2010-2020": diebold_mariano(common[yr <= 2020], m, f"{m}_imp"),
                 "DM 2021-2025": diebold_mariano(common[yr >= 2021], m, f"{m}_imp")})
same_obs = pd.DataFrame(rows).set_index("model")
print(same_obs.round(3).to_string())

### On the recovered observations

In [ ]:
key_c = pd.MultiIndex.from_frame(Pc[["symbol", "date"]])
key_i = pd.MultiIndex.from_frame(Pi[["symbol", "date"]])
recovered, retained = Pi[~key_i.isin(key_c)], Pi[key_i.isin(key_c)]
print(f"Recovered: {len(recovered):,} forecasts ({100*len(recovered)/len(Pi):.1f}%), "
      f"median market value {recovered['MV'].median():.1f} mn against {retained['MV'].median():.1f} mn retained\n")
rec = pd.DataFrame({
    "R2 recovered": annual_r2(recovered, MODELS).mean(),
    "R2 retained": annual_r2(retained, MODELS).mean(),
    "CS R2 recovered": pd.Series({m: 100 * r2_cross_section(recovered, m) for m in MODELS}),
    "CS R2 retained": pd.Series({m: 100 * r2_cross_section(retained, m) for m in MODELS})})
print("Models estimated on the imputed sample, 2010-2025 (%):")
print(rec.round(3).to_string())

## Selected hyperparameters

Depth, predictors per split, learning rate and number of trees selected in each test year.

In [ ]:
for s in SAMPLES:
    print(f"\n{s}:")
    print(R[s]["T"].set_index("year").to_string())

## Variable importance

In [ ]:
for s in SAMPLES:
    tab = pd.DataFrame({m: R[s]["imp"][m].mean(axis=1) for m in ["rf", "gbrt"]})
    tab = tab.sort_values("gbrt", ascending=False)
    print(f"\n{s} -- average share of gain importance (%):")
    print((100 * tab).round(1).head(15).to_string())

## Portfolio performance

In [ ]:
rows = []
for s in SAMPLES:
    for m in MODELS:
        ew, vw = decile_portfolios(R[s]["P"], m, ret="raw_target", weight="MV")
        for lab, port in [("equal", ew), ("value", vw)]:
            rows.append({"sample": s, "model": m, "weighting": lab, **portfolio_stats(port)})
perf = pd.DataFrame(rows)
print(perf.pivot_table(index="model", columns=["sample", "weighting"],
                       values=["mean_%", "t_stat", "sharpe_ann"]).round(2).reindex(MODELS).to_string())

for s in SAMPLES:
    for m in ["rf", "gbrt"]:
        ew, vw = decile_portfolios(R[s]["P"], m, ret="raw_target", weight="MV")
        print(f"\n{s} -- {m}, mean monthly excess return by decile (%):")
        print(pd.DataFrame({"equal": 100 * ew.mean(), "value": 100 * vw.mean()}).round(3).T.to_string())

## Save

In [ ]:
TAG = SAMPLES[0].replace("complete_case", "")
for s in SAMPLES:
    R[s]["P"].to_parquet(os.path.join(OUT_DIR, f"predictions_all_{s}.parquet"), index=False)
    R[s]["T"].to_csv(os.path.join(OUT_DIR, f"tuning_trees_{s}.csv"), index=False)
    for m in ["rf", "gbrt"]:
        R[s]["imp"][m].to_csv(os.path.join(OUT_DIR, f"importance_{m}_{s}.csv"))
for name, df in [("r2_avg_annual", avg), ("r2_pooled", pooled), ("r2_cross_section", cs),
                 ("r2_by_size", size_r2), ("predictive_slopes", ps_tab),
                 ("same_observations", same_obs), ("recovered", rec)]:
    df.to_csv(os.path.join(OUT_DIR, f"{name}_trees{TAG}.csv"))
perf.to_csv(os.path.join(OUT_DIR, f"portfolios_trees{TAG}.csv"), index=False)
for s in SAMPLES:
    dm[s].to_csv(os.path.join(OUT_DIR, f"diebold_mariano_trees_{s}.csv"))
print(f"Saved forecasts of all models, tuning, importance and result tables (suffix '{TAG}').")